In [44]:
import pandas as pd
df=pd.read_csv('dataset_incendi_FIRMS.csv')

KeyboardInterrupt: 

In [ ]:
%matplotlib inline

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")
plt.rcParams['figure.dpi'] = 120


In [ ]:
CSV_PATH = "dataset_incendi_FIRMS.csv"
CAMPIONE = 200_000

COL_LAT  = "latitude"
COL_LON  = "longitude"
COL_DATA = "acq_date"

FASCE = [
    ("2011–2015", 2011, 2015, "#4cc9f0", 0.35),
    ("2016–2020", 2016, 2020, "#f9c74f", 0.45),
    ("2021–2025", 2021, 2025, "#f94144", 0.55),
    ("2026",      2026, 2026, "#9d0208", 0.70),
]

# Colormap progressivi: freddo → caldo → critico → estremo
COLORMAPS = ["Blues", "YlOrBr", "OrRd", "Reds"]

BG_FIG  = "#0d1117"
BG_AX   = "#0d1b2a"
COL_PAE = "#1a2a3a"
COL_BOR = "#3a5a7a"
COL_GRI = "#2a3a4a"

BINS_LON = np.linspace(-180, 180, 300)   # risoluzione aumentata
BINS_LAT = np.linspace(-90,   90, 150)

In [ ]:

# ─── CARICAMENTO DATI ─────────────────────────────────────
def carica_dati(path, campione=None):
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"File non trovato: {path}")

    df = pd.read_csv(p, low_memory=False)
    df[COL_LAT] = pd.to_numeric(df[COL_LAT], errors="coerce")
    df[COL_LON] = pd.to_numeric(df[COL_LON], errors="coerce")
    df["_anno"] = pd.to_datetime(df[COL_DATA], errors="coerce").dt.year
    df = df.dropna(subset=[COL_LAT, COL_LON, "_anno"])
    df["_anno"] = df["_anno"].astype(int)

    if campione and len(df) > campione:
        df = df.sample(campione, random_state=42)

    print(f"Dati caricati: {len(df):,} righe | anni: {df['_anno'].min()}–{df['_anno'].max()}")
    return df

In [ ]:
def carica_confini():
    try:
        import geodatasets, geopandas as gpd
        world = gpd.read_file(geodatasets.get_path("naturalearth.land"))
        print("Confini geografici caricati.")
        return world
    except Exception as e:
        print(f"geopandas non disponibile ({e}), mappa senza confini.")
        return None  # ← era "N"

In [ ]:

# ─── HEATMAP DI DENSITÀ ───────────────────────────────────
def calcola_heatmap(sub):
    """Calcola e normalizza la matrice di densità 2D per un sottoinsieme."""
    H, _, _ = np.histogram2d(
        sub[COL_LON],
        sub[COL_LAT],
        bins=[BINS_LON, BINS_LAT]
    )
    H = H.T  # orientamento: lat sull'asse Y
    if H.max() > 0:
        # normalizzazione logaritmica: enfatizza aree sparse senza saturare i picchi
        H = np.log1p(H)
        H = H / H.max()
    return H


In [ ]:

# ─── DISEGNO PRINCIPALE ───────────────────────────────────
def disegna_rischio(df, world=None):
    """
    Genera la mappa spazio-temporale del rischio incendi.

    Parameters
    ----------
    df    : DataFrame con colonne latitude, longitude, _anno
    world : GeoDataFrame dei confini (opzionale)
    """
    fig, ax = plt.subplots(figsize=(22, 11))
    fig.patch.set_facecolor(BG_FIG)
    ax.set_facecolor(BG_AX)

    # — sfondo paesi —
    if world is not None:
        world.plot(ax=ax, color=COL_PAE, edgecolor=COL_BOR, linewidth=0.5, zorder=1)

    # — heatmap per fascia temporale —
    for i, (label, anno_min, anno_max, colore, alpha) in enumerate(FASCE):
        sub = df[(df["_anno"] >= anno_min) & (df["_anno"] <= anno_max)]

        if sub.empty:
            print(f"  Nessun dato per {label}, fascia saltata.")
            continue

        H = calcola_heatmap(sub)

        ax.imshow(
            H,
            extent=[-180, 180, -90, 90],
            origin="lower",
            cmap=COLORMAPS[i],
            alpha=alpha,
            vmin=0,
            vmax=1,
            interpolation="gaussian",   # bordi più morbidi
            zorder=2 + i                 # le epoche più recenti sopra
        )

        print(f"  {label}: {len(sub):,} punti · max densità cella: {H.max():.3f}")

    # — bordi sopra le heatmap —
    if world is not None:
        world.plot(ax=ax, color="none", edgecolor=COL_BOR, linewidth=0.4, zorder=10)

    # — griglia e assi —
    ax.set_xlim(-180, 180)
    ax.set_ylim(-90, 90)
    ax.grid(color=COL_GRI, linestyle="--", linewidth=0.3, zorder=0)
    ax.set_xlabel("Longitudine", color="#6a8aaa", fontsize=9)
    ax.set_ylabel("Latitudine",  color="#6a8aaa", fontsize=9)
    ax.tick_params(colors="#6a8aaa", labelsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor(COL_BOR)

    # — titolo —
    ax.set_title(
        "🔥 Mappa di Rischio Spazio-Temporale degli Incendi\n"
        "Densità e impatto nel tempo",
        color="white",
        fontsize=16,
        pad=14
    )

    # — legenda —
    handles = [
        mpatches.Patch(facecolor="#4cc9f0", edgecolor="white", linewidth=0.3,
                       label="2011–2015  (basso impatto)"),
        mpatches.Patch(facecolor="#f9c74f", edgecolor="white", linewidth=0.3,
                       label="2016–2020  (crescita)"),
        mpatches.Patch(facecolor="#f94144", edgecolor="white", linewidth=0.3,
                       label="2021–2025  (critico)"),
        mpatches.Patch(facecolor="#9d0208", edgecolor="white", linewidth=0.3,
                       label="2026         (alto rischio)"),
    ]
    leg = ax.legend(
        handles=handles,
        loc="lower left",
        bbox_to_anchor=(0.01, 0.01),
        facecolor=BG_AX,
        edgecolor=COL_BOR,
        labelcolor="white",
        fontsize=10,
        title="Evoluzione rischio",
        title_fontsize=11,
        framealpha=0.85,
    )
    leg.get_title().set_color("white")

    # — annotazione tendenza —
    fig.text(
        0.13, 0.88,
        "⬆ Intensificazione e espansione\ndel rischio incendi globale",
        color="#f94144",
        fontsize=11,
        fontweight="bold",
        ha="center",
        bbox=dict(boxstyle="round,pad=0.4", facecolor=BG_AX,
                  edgecolor="#f94144", alpha=0.8)
    )

    plt.tight_layout()
    return fig

In [ ]:

# ─── ESECUZIONE ───────────────────────────────────────────
df    = carica_dati(CSV_PATH, CAMPIONE)
world = carica_confini()

fig = disegna_rischio(df, world)
plt.savefig("mappa_rischio_incendi.png", dpi=150, bbox_inches="tight",
            facecolor=BG_FIG)
plt.show()